# 🌊 FlumenData — Dashboard Metrics Collector

Notebook này thu thập metrics từ MinIO, Spark, Delta Lake và ghi vào `metrics_db` trên PostgreSQL.

**Chạy lần đầu:** Đảm bảo đã tạo schema bằng `sql/01_create_metrics_db.sql`

## 0. Setup — Cài thư viện & cấu hình

In [ ]:
import subprocess, sys

# Cài packages cần thiết trong JupyterLab
for pkg in ["minio", "psycopg2-binary", "apscheduler"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("✅ Packages ready")

In [ ]:
import os

# ── Cấu hình — điều chỉnh nếu khác ──────────────────────────
os.environ.setdefault("MINIO_ENDPOINT",   "minio:9000")        # trong Docker network
os.environ.setdefault("MINIO_ACCESS_KEY", "admin")
os.environ.setdefault("MINIO_SECRET_KEY", "admin123")
os.environ.setdefault("SPARK_MASTER_URL", "http://spark-master:8080")
os.environ.setdefault("PG_HOST",          "postgres")
os.environ.setdefault("PG_PORT",          "5432")
os.environ.setdefault("PG_USER",          "admin")
os.environ.setdefault("PG_PASSWORD",      "admin123")
os.environ.setdefault("PG_DATABASE",      "metrics_db")

print("✅ Environment configured")

## 1. Tạo schema (chỉ cần chạy 1 lần)

In [ ]:
import psycopg2

# Đọc và chạy SQL schema
with open("/home/jovyan/work/sql/01_create_metrics_db.sql") as f:
    sql = f.read()

# Kết nối postgres (default db) để tạo metrics_db
conn = psycopg2.connect(
    host=os.environ["PG_HOST"],
    port=os.environ["PG_PORT"],
    user=os.environ["PG_USER"],
    password=os.environ["PG_PASSWORD"],
    dbname="postgres"  # connect to default db first
)
conn.autocommit = True
with conn.cursor() as cur:
    try:
        cur.execute("CREATE DATABASE metrics_db")
        print("✅ metrics_db created")
    except Exception as e:
        print(f"ℹ️  {e}")
conn.close()

# Áp schema vào metrics_db
conn = psycopg2.connect(
    host=os.environ["PG_HOST"],
    port=os.environ["PG_PORT"],
    user=os.environ["PG_USER"],
    password=os.environ["PG_PASSWORD"],
    dbname="metrics_db"
)
# Chỉ chạy phần CREATE TABLE / INDEX / VIEW (bỏ CREATE DATABASE)
schema_sql = "\n".join(
    line for line in sql.split("\n")
    if not line.strip().startswith("CREATE DATABASE")
    and not line.strip().startswith("\\c")
    and not line.strip().startswith("\\echo")
)
with conn.cursor() as cur:
    cur.execute(schema_sql)
conn.commit()
conn.close()
print("✅ Schema applied to metrics_db")

## 2. Test kết nối từng service

In [ ]:
from minio import Minio
import requests

# MinIO
try:
    mc = Minio(os.environ["MINIO_ENDPOINT"],
               access_key=os.environ["MINIO_ACCESS_KEY"],
               secret_key=os.environ["MINIO_SECRET_KEY"],
               secure=False)
    buckets = mc.list_buckets()
    print(f"✅ MinIO: {len(buckets)} buckets — {[b.name for b in buckets]}")
except Exception as e:
    print(f"❌ MinIO: {e}")

# Spark REST
try:
    r = requests.get(f"{os.environ['SPARK_MASTER_URL']}/api/v1/applications", timeout=5)
    apps = r.json()
    print(f"✅ Spark: {len(apps)} applications found")
except Exception as e:
    print(f"❌ Spark REST: {e}")

# PostgreSQL
try:
    conn = psycopg2.connect(
        host=os.environ["PG_HOST"], port=os.environ["PG_PORT"],
        user=os.environ["PG_USER"], password=os.environ["PG_PASSWORD"],
        dbname="metrics_db"
    )
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM storage_snapshots")
        n = cur.fetchone()[0]
    conn.close()
    print(f"✅ PostgreSQL metrics_db: storage_snapshots has {n} rows")
except Exception as e:
    print(f"❌ PostgreSQL: {e}")

## 3. Collector — Chạy 1 lần

In [ ]:
import sys
sys.path.insert(0, "/home/jovyan/work/collector")
from metrics_collector import run_once

# Lấy SparkSession đang chạy (tránh tạo mới)
try:
    from pyspark.sql import SparkSession
    spark = SparkSession.getActiveSession()
    print(f"✅ Using existing Spark session: {spark.version}")
except Exception:
    spark = None
    print("ℹ️  No active Spark session — Delta/Catalog stats will be skipped")

summary = run_once(spark=spark)
print("\nSummary:", summary)

## 4. Scheduler — Chạy định kỳ mỗi 15 phút

In [ ]:
from apscheduler.schedulers.background import BackgroundScheduler
import logging
logging.getLogger("apscheduler").setLevel(logging.WARNING)

scheduler = BackgroundScheduler()
scheduler.add_job(
    lambda: run_once(spark=spark),
    trigger="interval",
    minutes=15,
    id="flumen_collector",
    replace_existing=True
)
scheduler.start()
print("✅ Scheduler started — collecting every 15 minutes")
print("   Next run:", scheduler.get_job("flumen_collector").next_run_time)
print("   To stop: scheduler.shutdown()")

## 5. Xem kết quả nhanh

In [ ]:
import pandas as pd
import psycopg2

conn = psycopg2.connect(
    host=os.environ["PG_HOST"], port=os.environ["PG_PORT"],
    user=os.environ["PG_USER"], password=os.environ["PG_PASSWORD"],
    dbname="metrics_db"
)

print("\n=== Storage by Layer ===")
df_storage = pd.read_sql("SELECT * FROM v_storage_by_layer ORDER BY size_bytes DESC", conn)
display(df_storage)

print("\n=== Top Tables by Size ===")
df_tables = pd.read_sql("SELECT * FROM v_top_tables_by_size LIMIT 10", conn)
display(df_tables)

print("\n=== Pipeline Health (last 24h) ===")
df_pipeline = pd.read_sql("SELECT * FROM v_pipeline_health", conn)
display(df_pipeline)

print("\n=== Small Files Alert ===")
df_small = pd.read_sql(
    "SELECT * FROM v_small_files_alert WHERE status = 'small_files' LIMIT 20", conn
)
display(df_small)

conn.close()